# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EarlPly/Start-W1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering (continued from ML-02 and ML-03). This notebook uses only the pseudonymized FlyRank warehouse. It describes observed search behavior and a possible editorial review workflow; it does not infer Google's algorithm or the causal effect of an edit.


## 1. Unit of analysis + time window — the five contract answers

1. **One row means what?** The source fact has one row per `report_date × client_hash_id × content_hash_id`. My derived feature frame has one row per pseudonymized content page as of **2026-03-14**. IDs are grouping keys, never learned features.
2. **Which table?** Only `fact_content_daily_performance`, restricted to its `month=2026-03` partition. I use GSC rows only when `gsc_data_available IS TRUE`.
3. **Which window?** March 1–14 supplies features known at the decision moment. March 15–17 is a gap. March 18–31 supplies a later observed outcome for the leakage exercise. June 2026, the final month, stays sealed.
4. **What output or target?** The lane's output will be **descriptive search-performance archetypes** for editorial triage, with no supervised cluster label. For this week's leakage demonstration only, `observed_decline_next14` marks whether later 14-day impressions fell by more than 20% versus the prior 14 days. That is an observed-outcome proxy, not proof that an edit would help.
5. **What is deliberately excluded?** Future impressions and `future_change_pct` are excluded from honest features because they are unavailable on March 14 and define the diagnostic outcome. GA4 engagement is also outside this five-feature slice because its availability begins at different times across clients.

**Human action:** an editor reviews a group's page examples and privately checks page/query context before choosing title/snippet review, refresh investigation, or monitoring. This notebook never recommends an automatic prune or redirect.


In [1]:
import importlib.util
import os
import subprocess
import sys

for module_name, package_name in [('duckdb', 'duckdb>=1.0'), ('sklearn', 'scikit-learn>=1.4')]:
    if importlib.util.find_spec(module_name) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package_name], check=True)

import duckdb
import pandas as pd

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except ImportError:
        pass
if not hf_token or not hf_token.startswith('hf_'):
    raise RuntimeError('Set a Hugging Face Read token as Colab Secret HF_TOKEN (with notebook access) or environment variable HF_TOKEN.')

con = duckdb.connect()
try:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + hf_token.replace("'", "''") + "')")
except Exception as exc:
    raise RuntimeError(str(exc).replace(hf_token, '[redacted]')) from None
del hf_token

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('Source: FlyRank warehouse, March 2026 partition; token was not displayed or stored in the notebook.')


Source: FlyRank warehouse, March 2026 partition; token was not displayed or stored in the notebook.


## 2. Fields: feature / label / context / excluded

Exactly **five** proposed features describe a page's observed search behavior at the March 14 decision point:

| Feature | Meaning | Available when? |
|---|---|---|
| `log_prior_impressions` | log(1 + GSC impressions) | Known from March 1–14, before the decision. |
| `prior_ctr_pct` | clicks ÷ impressions × 100 | Both counts were observed by March 14; the unit is percentage points. |
| `prior_avg_position` | impression-weighted average search position | Built only from GSC position sums available by March 14; lower is better. |
| `prior_active_days` | days with at least one impression | Counts March 1–14 only. |
| `prior_week_change_pct` | change in impressions, March 8–14 versus March 1–7 | Both seven-day windows ended by March 14. |

**Label/proxy:** `observed_decline_next14` is the later-window diagnostic outcome used only in the trap experiment. Clustering itself has no supervised target. **Context:** `report_date`, `client_hash_id`, `content_hash_id`, and availability flags define windows, grouping, and a client-held-out split; none enters the model. **Excluded:** March 18–31 impressions and `future_change_pct` (future leakage), `gsc_data_available IS NOT TRUE` rows (missing search measurement), GA4 metrics (different coverage), and the June final-month sample (sealed test period).


In [2]:
feature_contract = pd.DataFrame({
    'feature': ['log_prior_impressions', 'prior_ctr_pct', 'prior_avg_position',
                'prior_active_days', 'prior_week_change_pct'],
    'latest_input_date': ['2026-03-14'] * 5,
    'role': ['feature'] * 5,
})
assert len(feature_contract) == 5
display(feature_contract)


,feature,latest_input_date,role
0,log_prior_impressions,2026-03-14,feature
1,prior_ctr_pct,2026-03-14,feature
2,prior_avg_position,2026-03-14,feature
3,prior_active_days,2026-03-14,feature
4,prior_week_change_pct,2026-03-14,feature


## 3. Verify with three small warehouse queries

Each query below reads **only March 2026**. Query 1 tests the stated daily grain. Query 2 measures the partition's count and date span. Query 3 checks GSC/GA4 availability, including the exact `IS TRUE` filter used for the lane slice. Counts here are observations, not estimates from the dataset guide.


In [3]:
# Verification query 1 / 3 — duplicate source-grain keys, limited to five examples.
q_grain = f'''
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS copies
FROM {MARCH}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
'''
grain_violations = con.sql(q_grain).df()
print(f'Duplicate date × client × content groups returned: {len(grain_violations)}')
display(grain_violations)
assert grain_violations.empty, 'Source daily grain is not unique; stop before aggregating.'


Duplicate date × client × content groups returned: 0


,report_date,client_hash_id,content_hash_id,copies


In [4]:
# Verification query 2 / 3 — actual size and dates of this one-month slice.
q_extent = f'''
SELECT COUNT(*) AS fact_rows,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items
FROM {MARCH}
'''
extent = con.sql(q_extent).df()
display(extent)
assert str(extent.loc[0, 'first_date'])[:10] == '2026-03-01'
assert str(extent.loc[0, 'last_date'])[:10] == '2026-03-31'


,fact_rows,first_date,last_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [5]:
# Verification query 3 / 3 — available search data and missingness.
q_availability = f'''
SELECT COUNT(*) AS fact_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_true_rows,
       COUNT(DISTINCT content_hash_id) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_content_items,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND gsc_sum_position IS NULL) AS gsc_rows_missing_position
FROM {MARCH}
'''
availability = con.sql(q_availability).df()
display(availability)
assert int(availability.loc[0, 'gsc_true_rows']) < int(availability.loc[0, 'fact_rows'])


,fact_rows,gsc_true_rows,gsc_not_true_rows,ga4_true_rows,both_true_rows,gsc_content_items,gsc_rows_missing_position
0,9841378,3611061,6230317,413966,364347,176738,0


### Five-feature page frame

Now aggregate the same mid-panel month to one row per page. The honest features stop on March 14. I require all 14 search-available days in both the prior and later windows, at least 100 prior impressions, and a nonzero first week so the ratios are defined. This is a deliberately measurable subset, not every page in the warehouse. Missing or zero position is left missing rather than treated as rank zero.


In [6]:
# Feature-building query; separate from the three contract-verification queries.
q_page_frame = f'''
WITH daily AS (
    SELECT report_date, client_hash_id, content_hash_id,
           COALESCE(gsc_impressions, 0) AS impressions,
           COALESCE(gsc_clicks, 0) AS clicks,
           gsc_sum_position
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
), per_page AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS prior_days,
           COUNT(DISTINCT report_date) FILTER (WHERE report_date BETWEEN DATE '2026-03-18' AND DATE '2026-03-31') AS future_days,
           SUM(impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS prior_impressions,
           SUM(clicks) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS prior_clicks,
           SUM(gsc_sum_position) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14') AS prior_sum_position,
           SUM(impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AND gsc_sum_position IS NOT NULL) AS position_covered_impressions,
           COUNT(*) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AND impressions > 0) AS prior_active_days,
           SUM(impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07') AS week1_impressions,
           SUM(impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-14') AS week2_impressions,
           SUM(impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-18' AND DATE '2026-03-31') AS future_impressions
    FROM daily
    GROUP BY 1, 2
)
SELECT client_hash_id, content_hash_id,
       LN(1 + prior_impressions) AS log_prior_impressions,
       100.0 * prior_clicks / prior_impressions AS prior_ctr_pct,
       prior_sum_position * 1.0 / position_covered_impressions AS prior_avg_position,
       prior_active_days,
       100.0 * (week2_impressions - week1_impressions) / week1_impressions AS prior_week_change_pct,
       100.0 * (future_impressions - prior_impressions) / prior_impressions AS future_change_pct,
       CAST(future_impressions < 0.8 * prior_impressions AS INTEGER) AS observed_decline_next14
FROM per_page
WHERE prior_days = 14 AND future_days = 14
  AND prior_impressions >= 100 AND week1_impressions > 0
  AND position_covered_impressions > 0
'''
page_frame = con.sql(q_page_frame).df()
honest_features = feature_contract['feature'].tolist()
X_honest = page_frame[honest_features].copy()
assert len(honest_features) == 5 and page_frame.content_hash_id.is_unique
print(f'Eligible page-level rows: {len(page_frame):,}; distinct clients: {page_frame.client_hash_id.nunique():,}')
print('Observed later-window decline rate: {:.1f}%'.format(100 * page_frame.observed_decline_next14.mean()))
display(page_frame[honest_features + ['observed_decline_next14']].head(8))


Eligible page-level rows: 57,459; distinct clients: 32
Observed later-window decline rate: 36.6%


,log_prior_impressions,prior_ctr_pct,prior_avg_position,prior_active_days,prior_week_change_pct,observed_decline_next14
0,5.043425,0.000000,6.863636,14,-37.894737,1
1,6.013715,0.735294,2.759804,14,21.739130,0
2,5.283204,1.020408,7.551020,14,-60.000000,1
3,6.466145,0.000000,8.464174,14,-28.342246,0
4,7.288928,0.000000,9.124402,14,-13.867684,1
5,5.826000,0.295858,6.526627,14,-31.840796,0
6,5.739793,1.290323,6.351613,14,84.403670,0
7,4.812184,0.819672,24.204918,14,39.215686,0


### Deliberate leakage trap, then removal

For this **diagnostic classification check only**, fit the same small tree on the five honest features and then on those features plus **one forbidden column**, `future_change_pct`. The latter is calculated from the March 18–31 outcome and therefore carries the answer. A client-held-out split keeps the comparison from merely memorizing the same clients. A near-perfect leaky score is evidence of a broken evaluation, not a better model. The clustering lane remains unsupervised.


In [7]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

y = page_frame['observed_decline_next14'].astype(int)
groups = page_frame['client_hash_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X_honest, y, groups))
assert y.iloc[test_idx].nunique() == 2, 'Held-out clients need both outcome classes.'

def quick_auc(frame):
    model = make_pipeline(SimpleImputer(strategy='median'),
                          DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=42))
    model.fit(frame.iloc[train_idx], y.iloc[train_idx])
    return roc_auc_score(y.iloc[test_idx], model.predict_proba(frame.iloc[test_idx])[:, 1])

honest_auc = quick_auc(X_honest)
X_leaky = X_honest.assign(future_change_pct=page_frame['future_change_pct'])
leaky_auc = quick_auc(X_leaky)
del X_leaky
assert list(X_honest.columns) == honest_features
print(f'Client-held-out ROC-AUC, five honest features: {honest_auc:.3f}')
print(f'Client-held-out ROC-AUC, with one future-derived leak: {leaky_auc:.3f}')
print('Removed the future-derived column; retained feature count:', X_honest.shape[1])


Client-held-out ROC-AUC, five honest features: 0.624
Client-held-out ROC-AUC, with one future-derived leak: 1.000
Removed the future-derived column; retained feature count: 5


## 4. Data limits and decision boundary

This is a **measured subset** of one month. Clients enter GSC at different dates, so rows with unavailable tracking cannot be read as zero traffic; query 3 shows exactly how many March rows survive `IS TRUE`. Full 14-day coverage and a 100-impression floor further narrow the sample, which may favor established pages and clients. The current five features omit GA4 engagement, content age, and query context, so any future clusters here describe **search-performance patterns**, not full editorial or semantic archetypes. A future decline is observed after the decision point, but it is not evidence that a particular edit caused recovery. No raw URLs, client names, or private queries are used.


In [8]:
march_rows = int(extent.loc[0, 'fact_rows'])
gsc_rows = int(availability.loc[0, 'gsc_true_rows'])
print(f'March fact rows: {march_rows:,}; GSC-available rows: {gsc_rows:,} ({gsc_rows/march_rows:.1%}).')
print(f'Feature-frame pages after coverage, volume, and position checks: {len(page_frame):,}.')
print('The frame is for descriptive triage and a leakage demonstration; no intervention benefit is estimated.')


March fact rows: 9,841,378; GSC-available rows: 3,611,061 (36.7%).
Feature-frame pages after coverage, volume, and position checks: 57,459.
The frame is for descriptive triage and a leakage demonstration; no intervention benefit is estimated.


## 5. Self-check

- [x] Five plain-language contract answers are given in section 1.
- [x] Exactly three verification queries above have visible outputs; availability uses `IS TRUE`.
- [x] The page frame has five time-safe features, each with an availability explanation.
- [x] One future-derived leak visibly raises the quick score, then is removed.
- [x] The notebook ran top to bottom; outputs contain no client names, URLs, or private queries.
- [x] Claims are observed and directional; the March slice's coverage limitation is named.
- [x] This executed notebook is committed under `work/notebooks/`; the repository URL is the submission link.
